In [ ]:
#Cell 1 — Hubungkan Google Drive dan siapkan folder

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/BiLSTM-CRF")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MAPPING_DIR = PROJECT_ROOT / "mappings"
EVALUATION_DIR = PROJECT_ROOT / "evaluation"

MODEL_DIRS = [
    PROJECT_ROOT / "models" / "baseline",
    PROJECT_ROOT / "models" / "pretrained",
    PROJECT_ROOT / "models" / "finetuned",
]

FOLDERS = [
    RAW_DIR,
    PROCESSED_DIR,
    MAPPING_DIR,
    EVALUATION_DIR,
    PROJECT_ROOT / "notebooks",
    *MODEL_DIRS,
]

for folder in FOLDERS:
    folder.mkdir(parents=True, exist_ok=True)

FATHUL_PATH = RAW_DIR / "dataset.csv"
TASHKEELA_PATH = RAW_DIR / "tashkeela_sampled.csv"

print("Project root :", PROJECT_ROOT)
print("Fathul Qorib :", FATHUL_PATH.exists())
print("Tashkeela    :", TASHKEELA_PATH.exists())

if not FATHUL_PATH.exists() or not TASHKEELA_PATH.exists():
    raise FileNotFoundError(
        "Pastikan dataset.csv dan tashkeela_sampled.csv tersedia di data/raw."
    )

print("✓ Cell 1 berhasil dijalankan.")

In [ ]:
#Cell 2 — Import library dan konfigurasi label
import json
import random
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
from IPython.display import display

# Hasil pengacakan tetap sama setiap dijalankan
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Delapan tanda harakat yang digunakan
HARAKAT = set("ًٌٍَُِّْ")
DAGGER_ALIF = "\u0670"

ALL_DIACRITICS_RE = re.compile(r"[\u064B-\u065F\u0670]")
ARABIC_LETTER_RE = re.compile(
    r"[\u0621-\u063A\u0641-\u064A\u066E-\u066F"
    r"\u0671-\u06D3\u06FA-\u06FC]"
)

# O berarti karakter tidak memiliki harakat
LABELS = [
    "O",
    "َ", "ُ", "ِ", "ْ",
    "ً", "ٌ", "ٍ",
    "ّ",
    "َ|ّ", "ُ|ّ", "ِ|ّ",
    "ً|ّ", "ٌ|ّ", "ٍ|ّ",
]

label2id = {label: index for index, label in enumerate(LABELS)}
id2label = {index: label for label, index in label2id.items()}

assert len(LABELS) == 15
assert len(set(LABELS)) == 15

print("Random seed :", SEED)
print("Jumlah label:", len(LABELS))
print("✓ Cell 2 berhasil dijalankan.")

In [ ]:
#Cell 3 — Fungsi normalisasi teks Arab

def normalize_arabic(text):
    # NFC menyamakan urutan Unicode harakat
    text = unicodedata.normalize("NFC", str(text))
    text = text.replace("ـ", "")
    text = text.replace(DAGGER_ALIF, "")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def remove_orphan_harakat(text):
    # Menghapus harakat yang tidak menempel pada huruf Arab
    output = []
    last_base_is_arabic = False
    removed = 0

    for char in text:
        if char in HARAKAT:
            if last_base_is_arabic:
                output.append(char)
            else:
                removed += 1
            continue

        output.append(char)
        last_base_is_arabic = bool(ARABIC_LETTER_RE.fullmatch(char))

    return "".join(output), removed


def clean_diacritized_text(text):
    text = normalize_arabic(text)
    text, removed = remove_orphan_harakat(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text, removed


def remove_diacritics(text):
    text = ALL_DIACRITICS_RE.sub("", text)
    return re.sub(r"\s+", " ", text).strip()


def find_unsupported_marks(text):
    return {
        char
        for char in text
        if unicodedata.combining(char) and char not in HARAKAT
    }


# Pengujian singkat fungsi
sample = "اللّٰهُ"
clean_sample, removed_sample = clean_diacritized_text(sample)

assert clean_sample == "اللّهُ"
assert remove_diacritics(clean_sample) == "الله"

print("Sebelum :", sample)
print("Sesudah :", clean_sample)
print("Gundul  :", remove_diacritics(clean_sample))
print("✓ Cell 3 berhasil dijalankan.")

In [ ]:
#Cell 4 — Membaca dan memvalidasi data mentah

fathul_raw = pd.read_csv(
    FATHUL_PATH,
    encoding="utf-8-sig",
    dtype={"page": int},
)

tashkeela_raw = pd.read_csv(
    TASHKEELA_PATH,
    encoding="utf-8-sig",
)

# Periksa kolom wajib
fathul_columns = {"page", "text_diacritized", "status"}
tashkeela_columns = {"teks_gundul", "teks_harakat"}

missing_fathul = fathul_columns - set(fathul_raw.columns)
missing_tashkeela = tashkeela_columns - set(tashkeela_raw.columns)

if missing_fathul:
    raise ValueError(f"Kolom Fathul Qorib tidak lengkap: {missing_fathul}")

if missing_tashkeela:
    raise ValueError(f"Kolom Tashkeela tidak lengkap: {missing_tashkeela}")

# Halaman 146, 150, dan 151 adalah halaman peta
EXPECTED_PAGES = set(range(19, 351)) - {146, 150, 151}
actual_pages = set(fathul_raw["page"])

assert len(fathul_raw) == 329
assert actual_pages == EXPECTED_PAGES
assert fathul_raw["page"].is_unique
assert fathul_raw["status"].eq("success").all()
assert fathul_raw["text_diacritized"].notna().all()

assert len(tashkeela_raw) == 7000
assert tashkeela_raw[list(tashkeela_columns)].notna().all().all()
assert not tashkeela_raw.duplicated().any()

audit_awal = pd.DataFrame([
    {
        "dataset": "Fathul Qorib",
        "baris": len(fathul_raw),
        "data_kosong": int(
            fathul_raw["text_diacritized"].astype(str).str.strip().eq("").sum()
        ),
        "duplikat": int(fathul_raw.duplicated().sum()),
    },
    {
        "dataset": "Tashkeela",
        "baris": len(tashkeela_raw),
        "data_kosong": int(
            tashkeela_raw[["teks_gundul", "teks_harakat"]]
            .astype(str)
            .apply(lambda column: column.str.strip().eq(""))
            .any(axis=1)
            .sum()
        ),
        "duplikat": int(tashkeela_raw.duplicated().sum()),
    },
])

display(audit_awal)

print("Halaman Fathul Qorib :", len(actual_pages))
print("Baris Tashkeela      :", len(tashkeela_raw))
print("✓ Cell 4 berhasil dijalankan.")

In [ ]:
#Cell 5 — Membersihkan teks Fathul Qorib

fathul = fathul_raw[["page", "text_diacritized"]].copy()

# Pastikan koreksi OCR halaman 20 sudah tersedia
page_20_text = fathul.loc[
    fathul["page"].eq(20),
    "text_diacritized",
].iloc[0]

assert "وَيُشْتَهَرُ أَيْضًا" in page_20_text, (
    "Koreksi OCR halaman 20 belum ditemukan."
)

tatweel_count = int(
    fathul["text_diacritized"].str.count("ـ").sum()
)

dagger_alif_count = int(
    fathul["text_diacritized"].str.count(DAGGER_ALIF).sum()
)

cleaned_texts = []
orphan_count = 0
unsupported_marks = Counter()

for text in fathul["text_diacritized"]:
    cleaned_text, removed = clean_diacritized_text(text)

    cleaned_texts.append(cleaned_text)
    orphan_count += removed
    unsupported_marks.update(
        find_unsupported_marks(cleaned_text)
    )

if unsupported_marks:
    raise ValueError(
        f"Tanda Unicode tidak didukung: {dict(unsupported_marks)}"
    )

fathul["text_diacritized"] = cleaned_texts
fathul["text_undiacritized"] = (
    fathul["text_diacritized"].map(remove_diacritics)
)

assert fathul["text_diacritized"].str.strip().ne("").all()
assert fathul["text_undiacritized"].str.strip().ne("").all()

assert fathul.apply(
    lambda row: remove_diacritics(
        row["text_diacritized"]
    ) == row["text_undiacritized"],
    axis=1,
).all()

print("Halaman diproses       :", len(fathul))
print("Koreksi halaman 20     : terverifikasi")
print("Tatweel dihapus        :", tatweel_count)
print("Dagger alif dihapus    :", dagger_alif_count)
print("Harakat yatim dihapus  :", orphan_count)
print("✓ Cell 5 berhasil dijalankan.")

In [ ]:
#Cell 6 — Membagi Fathul Qorib maksimal 300 karakter

MAX_LEN = 300


def base_length(text):
    return len(remove_diacritics(text))


def split_long_word(word, max_len):
    chunks = []
    current = []
    current_length = 0

    for char in word:
        if char not in HARAKAT and current_length >= max_len:
            chunks.append("".join(current))
            current = []
            current_length = 0

        current.append(char)

        if char not in HARAKAT:
            current_length += 1

    if current:
        chunks.append("".join(current))

    return chunks


def split_text(text, max_len=MAX_LEN):
    words = text.split()
    chunks = []
    current = ""

    for word in words:
        candidate = word if not current else f"{current} {word}"

        if base_length(candidate) <= max_len:
            current = candidate
            continue

        if current:
            chunks.append(current)
            current = ""

        if base_length(word) <= max_len:
            current = word
        else:
            word_chunks = split_long_word(word, max_len)
            chunks.extend(word_chunks[:-1])
            current = word_chunks[-1]

    if current:
        chunks.append(current)

    return chunks


fathul_rows = []

for row in fathul.itertuples(index=False):
    page_chunks = split_text(row.text_diacritized)

    for chunk_number, target_text in enumerate(page_chunks, start=1):
        source_text = remove_diacritics(target_text)

        fathul_rows.append({
            "page": int(row.page),
            "chunk_id": f"FQ_{int(row.page):03d}_{chunk_number:02d}",
            "text_undiacritized": source_text,
            "text_diacritized": target_text,
            "length": len(source_text),
        })


fathul_processed = pd.DataFrame(fathul_rows)

assert fathul_processed["length"].between(1, MAX_LEN).all()
assert fathul_processed["chunk_id"].is_unique

assert fathul_processed.apply(
    lambda row: remove_diacritics(
        row["text_diacritized"]
    ) == row["text_undiacritized"],
    axis=1,
).all()

# Pastikan tidak ada teks halaman yang hilang
for row in fathul.itertuples(index=False):
    reconstructed = " ".join(
        fathul_processed.loc[
            fathul_processed["page"].eq(row.page),
            "text_diacritized",
        ]
    )

    assert reconstructed == row.text_diacritized, (
        f"Teks halaman {row.page} tidak tersusun kembali dengan benar."
    )


print("Jumlah halaman  :", fathul_processed["page"].nunique())
print("Jumlah sequence :", len(fathul_processed))
print("Panjang minimum :", fathul_processed["length"].min())
print("Panjang rata-rata:", round(fathul_processed["length"].mean(), 2))
print("Panjang maksimum:", fathul_processed["length"].max())
print("✓ Cell 6 berhasil dijalankan.")

display(fathul_processed.head(3))

In [ ]:
#Cell 7 — Membersihkan dan memvalidasi Tashkeela

tashkeela_rows = []
orphan_count = 0
unsupported_marks = Counter()
mismatched_rows = []

for index, row in tashkeela_raw.iterrows():
    source_row = index + 1

    given_source = normalize_arabic(row["teks_gundul"])
    target_text, removed = clean_diacritized_text(
        row["teks_harakat"]
    )
    derived_source = remove_diacritics(target_text)

    orphan_count += removed
    unsupported_marks.update(
        find_unsupported_marks(target_text)
    )

    if given_source != derived_source:
        mismatched_rows.append({
            "source_row": source_row,
            "teks_gundul": given_source,
            "hasil_hapus_harakat": derived_source,
        })

    tashkeela_rows.append({
        "source_row": source_row,
        "sequence_id": f"TS_{source_row:05d}",
        "text_undiacritized": derived_source,
        "text_diacritized": target_text,
        "length": len(derived_source),
    })


if unsupported_marks:
    raise ValueError(
        f"Tanda Unicode tidak didukung: {dict(unsupported_marks)}"
    )

if mismatched_rows:
    display(pd.DataFrame(mismatched_rows).head())
    raise ValueError(
        f"Ditemukan {len(mismatched_rows)} pasangan Tashkeela yang tidak sesuai."
    )


tashkeela_processed = pd.DataFrame(tashkeela_rows)

duplicate_count = int(
    tashkeela_processed.duplicated(
        subset=["text_undiacritized"]
    ).sum()
)

if duplicate_count:
    tashkeela_processed = (
        tashkeela_processed
        .drop_duplicates(subset=["text_undiacritized"])
        .reset_index(drop=True)
    )

assert tashkeela_processed["length"].between(1, MAX_LEN).all()
assert tashkeela_processed["sequence_id"].is_unique

assert tashkeela_processed.apply(
    lambda row: remove_diacritics(
        row["text_diacritized"]
    ) == row["text_undiacritized"],
    axis=1,
).all()


print("Data awal             :", len(tashkeela_raw))
print("Data setelah diproses :", len(tashkeela_processed))
print("Harakat yatim dihapus :", orphan_count)
print("Duplikat dihapus      :", duplicate_count)
print("Panjang minimum       :", tashkeela_processed["length"].min())
print("Panjang maksimum      :", tashkeela_processed["length"].max())
print("✓ Cell 7 berhasil dijalankan.")

display(tashkeela_processed.head(3))

In [ ]:
#Cell 8 — Membagi data train, validation, dan test

# Fathul Qorib dibagi berdasarkan halaman: 80:10:10
pages = np.array(
    sorted(fathul_processed["page"].unique())
)

page_rng = np.random.default_rng(SEED)
page_rng.shuffle(pages)

n_pages = len(pages)
n_test_pages = round(n_pages * 0.10)
n_validation_pages = round(n_pages * 0.10)

test_pages = set(pages[:n_test_pages])
validation_pages = set(
    pages[n_test_pages:n_test_pages + n_validation_pages]
)
train_pages = set(
    pages[n_test_pages + n_validation_pages:]
)


def assign_fathul_split(page):
    if page in train_pages:
        return "train"
    if page in validation_pages:
        return "validation"
    return "test"


fathul_processed["split"] = (
    fathul_processed["page"].map(assign_fathul_split)
)

assert train_pages.isdisjoint(validation_pages)
assert train_pages.isdisjoint(test_pages)
assert validation_pages.isdisjoint(test_pages)

assert (
    fathul_processed.groupby("page")["split"].nunique().max()
    == 1
)


# Tashkeela hanya dibagi menjadi train dan validation: 90:10
tashkeela_rng = np.random.default_rng(SEED + 1)
shuffled_indices = tashkeela_rng.permutation(
    tashkeela_processed.index
)

n_tashkeela_validation = round(
    len(tashkeela_processed) * 0.10
)

tashkeela_validation_indices = set(
    shuffled_indices[:n_tashkeela_validation]
)

tashkeela_processed["split"] = np.where(
    tashkeela_processed.index.isin(
        tashkeela_validation_indices
    ),
    "validation",
    "train",
)


fathul_train = fathul_processed[
    fathul_processed["split"].eq("train")
].reset_index(drop=True)

fathul_validation = fathul_processed[
    fathul_processed["split"].eq("validation")
].reset_index(drop=True)

fathul_test = fathul_processed[
    fathul_processed["split"].eq("test")
].reset_index(drop=True)

tashkeela_train = tashkeela_processed[
    tashkeela_processed["split"].eq("train")
].reset_index(drop=True)

tashkeela_validation = tashkeela_processed[
    tashkeela_processed["split"].eq("validation")
].reset_index(drop=True)


fathul_split_summary = (
    fathul_processed.groupby("split")
    .agg(
        pages=("page", "nunique"),
        sequences=("chunk_id", "count"),
        characters=("length", "sum"),
    )
    .reset_index()
)

tashkeela_split_summary = (
    tashkeela_processed.groupby("split")
    .agg(
        sequences=("sequence_id", "count"),
        characters=("length", "sum"),
    )
    .reset_index()
)

print("Pembagian Fathul Qorib")
display(fathul_split_summary)

print("Pembagian Tashkeela")
display(tashkeela_split_summary)

print("✓ Cell 8 berhasil dijalankan.")

In [ ]:
#Cell 9 — Membentuk dan memvalidasi label karakter

def extract_chars_and_labels(text):
    chars = []
    labels = []
    index = 0

    while index < len(text):
        char = text[index]

        if char in HARAKAT:
            raise ValueError(
                f"Harakat tidak memiliki karakter dasar pada indeks {index}."
            )

        chars.append(char)
        marks = []
        next_index = index + 1

        while (
            next_index < len(text)
            and text[next_index] in HARAKAT
        ):
            mark = text[next_index]

            if mark not in marks:
                marks.append(mark)

            next_index += 1

        label = "|".join(marks) if marks else "O"

        if label not in label2id:
            raise ValueError(
                f"Label tidak didukung: {repr(label)}"
            )

        labels.append(label)
        index = next_index

    return chars, labels


def reconstruct_text(chars, labels):
    output = []

    for char, label in zip(chars, labels):
        output.append(char)

        if label != "O":
            output.extend(label.split("|"))

    return "".join(output)


def validate_labels(frame, dataset_name):
    distribution = Counter()

    for row in frame.itertuples(index=False):
        chars, labels = extract_chars_and_labels(
            row.text_diacritized
        )

        assert len(chars) == len(labels)
        assert "".join(chars) == row.text_undiacritized
        assert reconstruct_text(chars, labels) == (
            row.text_diacritized
        )

        distribution.update(labels)

    return pd.DataFrame({
        "label": LABELS,
        dataset_name: [
            distribution.get(label, 0)
            for label in LABELS
        ],
    })


fathul_label_counts = validate_labels(
    fathul_processed,
    "fathul_qorib",
)

tashkeela_label_counts = validate_labels(
    tashkeela_processed,
    "tashkeela",
)

label_distribution = fathul_label_counts.merge(
    tashkeela_label_counts,
    on="label",
    how="outer",
).fillna(0)

label_distribution["total"] = (
    label_distribution["fathul_qorib"]
    + label_distribution["tashkeela"]
)

assert len(label_distribution) == 15
assert label_distribution["total"].gt(0).all()

display(label_distribution)

print("Jumlah label tervalidasi:", len(label_distribution))
print("✓ Cell 9 berhasil dijalankan.")

In [ ]:
#Cell 10 — Membuat dan menyimpan mapping karakter serta label

# Mapping hanya dibuat dari data training
training_texts = pd.concat(
    [
        fathul_train["text_undiacritized"],
        tashkeela_train["text_undiacritized"],
    ],
    ignore_index=True,
)

training_characters = sorted(
    set("".join(training_texts))
)

char2id = {
    "<PAD>": 0,
    "<UNK>": 1,
}

for char in training_characters:
    if char not in char2id:
        char2id[char] = len(char2id)

id2char = {
    index: char
    for char, index in char2id.items()
}


def get_unknown_characters(frame):
    characters = set(
        "".join(frame["text_undiacritized"])
    )
    return sorted(
        characters - set(char2id.keys())
    )


unknown_summary = {
    "fathul_validation": get_unknown_characters(
        fathul_validation
    ),
    "fathul_test": get_unknown_characters(
        fathul_test
    ),
    "tashkeela_validation": get_unknown_characters(
        tashkeela_validation
    ),
}


mapping_files = {
    "char2id.json": char2id,
    "id2char.json": id2char,
    "label2id.json": label2id,
    "id2label.json": id2label,
}

for file_name, mapping in mapping_files.items():
    with open(
        MAPPING_DIR / file_name,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            mapping,
            file,
            ensure_ascii=False,
            indent=2,
        )


print("Jumlah karakter :", len(char2id))
print("Jumlah label    :", len(label2id))
print("ID padding      :", char2id["<PAD>"])
print("ID unknown      :", char2id["<UNK>"])

print("\nKarakter yang tidak dikenal:")
for split_name, characters in unknown_summary.items():
    print(f"{split_name:22}: {characters or 'Tidak ada'}")

print("\nFile mapping:")
for file_name in mapping_files:
    print("✓", MAPPING_DIR / file_name)

print("✓ Cell 10 berhasil dijalankan.")

In [ ]:
#Cell 11 — Menyimpan dataset processed dan konfigurasi

dataset_files = {
    "fathul_all.csv": fathul_processed,
    "fathul_train.csv": fathul_train,
    "fathul_validation.csv": fathul_validation,
    "fathul_test.csv": fathul_test,
    "tashkeela_all.csv": tashkeela_processed,
    "tashkeela_train.csv": tashkeela_train,
    "tashkeela_validation.csv": tashkeela_validation,
}

for file_name, frame in dataset_files.items():
    frame.to_csv(
        PROCESSED_DIR / file_name,
        index=False,
        encoding="utf-8-sig",
    )


label_distribution[
    ["fathul_qorib", "tashkeela", "total"]
] = label_distribution[
    ["fathul_qorib", "tashkeela", "total"]
].astype(int)

label_distribution.to_csv(
    PROCESSED_DIR / "label_distribution.csv",
    index=False,
    encoding="utf-8-sig",
)

fathul_split_summary.to_csv(
    PROCESSED_DIR / "fathul_split_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

tashkeela_split_summary.to_csv(
    PROCESSED_DIR / "tashkeela_split_summary.csv",
    index=False,
    encoding="utf-8-sig",
)


split_pages = {
    "train": sorted(map(int, train_pages)),
    "validation": sorted(map(int, validation_pages)),
    "test": sorted(map(int, test_pages)),
}

with open(
    PROCESSED_DIR / "fathul_split_pages.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        split_pages,
        file,
        ensure_ascii=False,
        indent=2,
    )


preprocessing_config = {
    "seed": SEED,
    "tashkeela_seed": SEED + 1,
    "max_length": MAX_LEN,
    "number_of_labels": len(LABELS),
    "dagger_alif_excluded": True,
    "excluded_pdf_pages": [146, 150, 151],
    "fathul_pages": int(
        fathul_processed["page"].nunique()
    ),
    "fathul_sequences": int(len(fathul_processed)),
    "tashkeela_sequences": int(
        len(tashkeela_processed)
    ),
    "fathul_train_pages": len(train_pages),
    "fathul_validation_pages": len(validation_pages),
    "fathul_test_pages": len(test_pages),
    "tashkeela_train_sequences": int(
        len(tashkeela_train)
    ),
    "tashkeela_validation_sequences": int(
        len(tashkeela_validation)
    ),
    "vocabulary_size": len(char2id),
}

with open(
    PROCESSED_DIR / "preprocessing_config.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        preprocessing_config,
        file,
        ensure_ascii=False,
        indent=2,
    )


# Pastikan semua file berhasil disimpan
saved_files = [
    *(PROCESSED_DIR / name for name in dataset_files),
    PROCESSED_DIR / "label_distribution.csv",
    PROCESSED_DIR / "fathul_split_summary.csv",
    PROCESSED_DIR / "tashkeela_split_summary.csv",
    PROCESSED_DIR / "fathul_split_pages.json",
    PROCESSED_DIR / "preprocessing_config.json",
]

for file_path in saved_files:
    assert file_path.exists()
    assert file_path.stat().st_size > 0


print("File dataset:")
for file_name, frame in dataset_files.items():
    print(f"✓ {file_name:27} {len(frame):5} baris")

print("\nFile konfigurasi:")
for file_path in saved_files[len(dataset_files):]:
    print("✓", file_path.name)

print("\nLokasi penyimpanan:")
print(PROCESSED_DIR)
print("✓ Cell 11 berhasil dijalankan.")

In [ ]:
#Cell 12 — Audit akhir preprocessing

fathul_check = pd.read_csv(
    PROCESSED_DIR / "fathul_all.csv",
    encoding="utf-8-sig",
)

tashkeela_check = pd.read_csv(
    PROCESSED_DIR / "tashkeela_all.csv",
    encoding="utf-8-sig",
)

with open(
    MAPPING_DIR / "label2id.json",
    encoding="utf-8",
) as file:
    saved_label2id = json.load(file)


assert len(fathul_check) == len(fathul_processed)
assert len(tashkeela_check) == len(tashkeela_processed)

assert fathul_check["page"].nunique() == 329
assert fathul_check["length"].between(1, MAX_LEN).all()
assert tashkeela_check["length"].between(1, MAX_LEN).all()

assert fathul_check["text_undiacritized"].notna().all()
assert fathul_check["text_diacritized"].notna().all()
assert tashkeela_check["text_undiacritized"].notna().all()
assert tashkeela_check["text_diacritized"].notna().all()

assert (
    fathul_check.groupby("page")["split"].nunique().max()
    == 1
)

assert set(saved_label2id.keys()) == set(LABELS)
assert len(saved_label2id) == 15

validate_labels(fathul_check, "fathul_check")
validate_labels(tashkeela_check, "tashkeela_check")

fathul_texts = set(
    fathul_check["text_undiacritized"]
)

tashkeela_texts = set(
    tashkeela_check["text_undiacritized"]
)

exact_overlap = len(
    fathul_texts.intersection(tashkeela_texts)
)

audit_final = {
    "status": "passed",
    "fathul_pages": int(
        fathul_check["page"].nunique()
    ),
    "fathul_sequences": int(len(fathul_check)),
    "tashkeela_sequences": int(
        len(tashkeela_check)
    ),
    "max_fathul_length": int(
        fathul_check["length"].max()
    ),
    "max_tashkeela_length": int(
        tashkeela_check["length"].max()
    ),
    "number_of_labels": len(saved_label2id),
    "vocabulary_size": len(char2id),
    "exact_dataset_overlap": exact_overlap,
}

with open(
    PROCESSED_DIR / "final_audit.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        audit_final,
        file,
        ensure_ascii=False,
        indent=2,
    )


print("Status audit          :", audit_final["status"])
print("Halaman Fathul Qorib :", audit_final["fathul_pages"])
print("Sequence Fathul Qorib:", audit_final["fathul_sequences"])
print("Sequence Tashkeela   :", audit_final["tashkeela_sequences"])
print("Jumlah label         :", audit_final["number_of_labels"])
print("Ukuran vocabulary    :", audit_final["vocabulary_size"])
print("Data sama antardataset:", exact_overlap)
print("✓ Notebook 01 selesai dan tervalidasi.")